# Hybrid Retrieval and Reranking

The final inference pipeline. This notebook implements a Two-Stage Architecture:

Retrieval (Recall Layer): A Hybrid approach combining Dense Vectors (Matryoshka 64d) and Neural Sparse (SPLADE + BM25) to maximize candidate coverage.

Reranking (Precision Layer): A Cross-Encoder (mxbai-rerank) re-scores the top candidates to maximize nDCG.

In [1]:
import sys, os, yaml, time, torch, pickle, dagshub, mlflow
import pandas as pd
from pathlib import Path
import numpy as np
from sentence_transformers import CrossEncoder

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    
from src.esci.sparse_retrievers import SPLADEFast, BM25Fast
from src.esci.system_b import build_candidates, rerank_candidates
from src.esci.metrics import compute_recall_metrics, compute_ndcg_metrics
from src.esci.artifacts import load_artifacts
from src.esci.mlflow import log_candidates_run, log_rerank_run

# Load Config
with open(project_root / "configs" / "esci.yaml") as f:
    cfg = yaml.safe_load(f)

## Setting up MLflow connection

In [2]:
dagshub.init(repo_owner='aminlasri', repo_name='Amazon-ESCI-Project', mlflow=True)

Accessing as aminlasri

Initialized MLflow to track repo "aminlasri/Amazon-ESCI-Project"

Repository aminlasri/Amazon-ESCI-Project initialized!

In [3]:
print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: https://dagshub.com/aminlasri/Amazon-ESCI-Project.mlflow


#### Here, we are setting the experiment name based on the chosen hyperparameters.

In [4]:
mlflow.set_experiment("XP_6_dim_64_rrf_k_60_weights_dense_1.0_splade_0.5_bm25_0.3")

<Experiment: artifact_location='mlflow-artifacts:/3c45d3f592654cba9f81e656963ea745', creation_time=1771624402167, experiment_id='13', last_update_time=1771624402167, lifecycle_stage='active', name='XP_6_dim_64_rrf_k_60_weights_dense_1.0_splade_0.5_bm25_0.3', tags={'mlflow.experimentKind': 'custom_model_development'}>

## STAGE 1: Hybrid Candidate Generation

In [5]:
# Strategy: Dense (64d) + SPLADE + BM25 (Hybrid)
print("⚡ Setting up Retrieval Phase...")

# Load Artifacts
prod_emb, qry_emb, prod_df, qry_df, pair_df = load_artifacts("us", "test", cfg)

# Build Ground Truth (q_rels)
print("📊 Building Ground Truth Map (q_rels)...")
q_rels = pair_df.groupby("query_id")["grade"].apply(list).to_dict()

# Load SPLADE (Keep loading this from disk, it's heavy and correct)
print("⚡ Loading SPLADE Matrix...")
splade_path = "../artifacts/systemA/splade_data.pt"
splade_data = torch.load(splade_path, map_location="cpu")

splade_loaded = SPLADEFast(
    cfg["sparse"]["splade_model"], 
    batch_size=cfg["sparse"]["batch_size"]
)
splade_loaded.doc_matrix = splade_data["doc_matrix"]
splade_loaded.pid_list   = splade_data["pid_list"]

# Align device
splade_loaded.doc_matrix = splade_loaded.doc_matrix.to(splade_loaded.device)
splade = splade_loaded

print("⚡ Loading BM25 Index...")
with open("../artifacts/systemA/bm25_retriever.pkl", "rb") as f:
    bm25_loaded = pickle.load(f)

bm25 = bm25_loaded

# Configure Retrieval
# We explicitly choose Dense + SPLADE + BM25
cfg["retrieval"]["sources"] = ["dense", "splade", "bm25"] 
dim_to_use = 64
sources = cfg["retrieval"]["sources"]

print(f"🔍 Retrieving Candidates (Matryoshka Dim {dim_to_use} + BM25 + SPLADE)...")

retrieval_df, retrieval_qps = build_candidates(
    cfg, 
    split="test", 
    override_dim=dim_to_use, 
    prebuilt_splade=splade, 
    prebuilt_bm25=bm25  
)

# Measure Retrieval Baseline
recall_results = compute_recall_metrics(retrieval_df, q_rels)
ndcg_results = compute_ndcg_metrics(retrieval_df, q_rels)

# Merge metrics
base_metrics = {**recall_results, **ndcg_results}

# save results to mlflow
log_candidates_run(cfg, dim_to_use, sources, base_metrics, retrieval_qps)

print(f"\n📊 Retrieval Dense + SPLADE + BM25 (Dim {dim_to_use}):")
print(f"   nDCG@10:    {base_metrics['nDCG@10']:.4f}")
print(f"   nDCG@20:    {base_metrics['nDCG@20']:.4f}")
print(f"   nDCG@50:    {base_metrics['nDCG@50']:.4f}")
print(f"   Recall@100: {base_metrics['Recall@100']:.4f}")
print(f"   Recall@200: {base_metrics['Recall@200']:.4f}")
print(f"   QPS:        {retrieval_qps:.2f}")

⚡ Setting up Retrieval Phase...
📊 Building Ground Truth Map (q_rels)...
⚡ Loading SPLADE Matrix...


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: naver/splade-cocondenser-ensembledistil
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⚡ Loading BM25 Index...
🔍 Retrieving Candidates (Matryoshka Dim 64 + BM25 + SPLADE)...
⚡ Using FAISS (Dim=64)
    -> Measuring QPS for 8908 queries...
🏃 View run candidates__dim64__dense_splade_bm25 at: https://dagshub.com/aminlasri/Amazon-ESCI-Project.mlflow/#/experiments/13/runs/21dd3acbeac641e98931b69d090bd7e5
🧪 View experiment at: https://dagshub.com/aminlasri/Amazon-ESCI-Project.mlflow/#/experiments/13

📊 Retrieval Dense + SPLADE + BM25 (Dim 64):
   nDCG@10:    0.4472
   nDCG@20:    0.4846
   nDCG@50:    0.5487
   Recall@100: 0.7393
   Recall@200: 0.8125
   QPS:        70.51


## What's the best batch size for this reranker?

In [6]:
def autotune_batch_size(ce, pairs, batch_sizes=(64, 128, 256, 512, 1024), warmup=1):
    device = ce.device
    results = []
    
    # small slice for tuning (enough to measure, not too slow)
    sample = pairs[:min(len(pairs), 2000)]
    
    for bs in batch_sizes:
        try:
            torch.cuda.empty_cache()
            # warmup (optional)
            for _ in range(warmup):
                _ = ce.predict(sample[:bs], batch_size=bs, show_progress_bar=False, num_workers=0)

            t0 = time.time()
            _ = ce.predict(sample, batch_size=bs, show_progress_bar=False, num_workers=0)
            dt = time.time() - t0
            
            qps = len(sample) / dt
            mem = torch.cuda.max_memory_allocated() / (1024**3) if torch.cuda.is_available() else 0.0
            results.append((bs, qps, mem))
            print(f"bs={bs:4d} | QPS={qps:8.1f} | peak_mem={mem:.2f} GB")
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"bs={bs:4d} | OOM")
            else:
                raise e

    # choose highest qps (or choose best within mem headroom)
    if not results:
        return None, []
    best = max(results, key=lambda x: x[1])
    return best[0], results


In [7]:
ce_model = cfg["cross_encoder_model"]

ce = CrossEncoder(
    ce_model,
    device="cuda",
    max_length= cfg["reranker"]["max_seq_length"]
)

pairs = list(zip(
    retrieval_df["query"].head(2000),
    retrieval_df["product_text_dense"].head(2000)
))

best_bs, res = autotune_batch_size(
    ce,
    pairs,
    batch_sizes=(4,8,16,32,64,128,256,512)
)

print("✅ Best batch_size:", best_bs)


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

The CrossEncoder.predict `num_workers` argument is deprecated and has no effect. It will be removed in a future version.
The CrossEncoder.predict `num_workers` argument is deprecated and has no effect. It will be removed in a future version.
The CrossEncoder.predict `num_workers` argument is deprecated and has no effect. It will be removed in a future version.
The CrossEncoder.predict `num_workers` argument is deprecated and has no effect. It will be removed in a future version.


bs=   4 | QPS=   163.2 | peak_mem=1.23 GB


The CrossEncoder.predict `num_workers` argument is deprecated and has no effect. It will be removed in a future version.
The CrossEncoder.predict `num_workers` argument is deprecated and has no effect. It will be removed in a future version.


bs=   8 | QPS=   274.1 | peak_mem=1.29 GB


...

bs= 128 | QPS=   231.7 | peak_mem=3.24 GB


The CrossEncoder.predict `num_workers` argument is deprecated and has no effect. It will be removed in a future version.
The CrossEncoder.predict `num_workers` argument is deprecated and has no effect. It will be removed in a future version.


bs= 256 | QPS=   230.6 | peak_mem=5.32 GB


The CrossEncoder.predict `num_workers` argument is deprecated and has no effect. It will be removed in a future version.


bs= 512 | QPS=    31.5 | peak_mem=9.48 GB
✅ Best batch_size: 8


## STAGE 2: Cross-Encoder Reranking

In [8]:
ce_model = cfg["cross_encoder_model"]

print(f"\n🚀 Reranking Top 50 Candidates with {ce_model}...")

# Rerank the top 50 candidates (Engineering tradeoff: Precision vs Latency)
reranked_df, rerank_qps = rerank_candidates(
    retrieval_df, 
    model_name=ce_model, 
    top_k_to_rerank=50,
    batch_size=best_bs,
    cfg=cfg)

# Measure Reranking Performance
final_metrics = compute_ndcg_metrics(reranked_df, q_rels)

# save results to mlflow
log_rerank_run(cfg, dim_to_use, sources, final_metrics, rerank_qps)

print(f"\n📊 Reranking Performance:")
print(f"   nDCG@10:    {final_metrics['nDCG@10']:.4f}")
print(f"   nDCG@20:    {final_metrics['nDCG@20']:.4f}")
print(f"   nDCG@50:    {final_metrics['nDCG@50']:.4f}")
print(f"   QPS:        {rerank_qps:.2f}")


🚀 Reranking Top 50 Candidates with mixedbread-ai/mxbai-rerank-base-v1...
🚀 Loading Cross-Encoder: mixedbread-ai/mxbai-rerank-base-v1 on CUDA... and Max length: 256


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

    -> Reranking 445400 pairs...
    -> Rerank Speed: 5.76 QPS
🏃 View run rerank__dim64__dense_splade_bm25 at: https://dagshub.com/aminlasri/Amazon-ESCI-Project.mlflow/#/experiments/13/runs/82efe5ca6d0d4083b3e418fd46b128d5
🧪 View experiment at: https://dagshub.com/aminlasri/Amazon-ESCI-Project.mlflow/#/experiments/13

📊 Reranking Performance:
   nDCG@10:    0.5081
   nDCG@20:    0.5395
   nDCG@50:    0.5854
   QPS:        5.76


## STEP 3: The Comparison

In [9]:
comparison = pd.DataFrame([
    {
        "Stage": "1. Retrieval (Hybrid 64d)",
        "nDCG@10": base_metrics["nDCG@10"],
        "nDCG@20": base_metrics["nDCG@20"],
        "nDCG@50": base_metrics["nDCG@50"],
        "Recall@100": base_metrics["Recall@100"],
        "Recall@200": base_metrics["Recall@200"],
        "QPS": retrieval_qps
    },
    {
        "Stage": "2. Reranking (Cross-Encoder)",
        "nDCG@10": final_metrics["nDCG@10"],
        "nDCG@20": final_metrics["nDCG@20"],
        "nDCG@50": final_metrics["nDCG@50"],
        "QPS": rerank_qps
    }
])

print("\n🏆 Final Pipeline Results:")
display(comparison)

# Calculate Accuracy Lift (nDCG is the main metric for ranking quality)
lift10 = (final_metrics['nDCG@10'] - base_metrics['nDCG@10']) / base_metrics['nDCG@10'] * 100
print(f"\n📈 Accuracy @10 Lift from Reranking: +{lift10:.1f}%")

lift20 = (final_metrics['nDCG@20'] - base_metrics['nDCG@20']) / base_metrics['nDCG@20'] * 100
print(f"\n📈 Accuracy @20 Lift from Reranking: +{lift20:.1f}%")

lift50 = (final_metrics['nDCG@50'] - base_metrics['nDCG@50']) / base_metrics['nDCG@50'] * 100
print(f"\n📈 Accuracy @50 Lift from Reranking: +{lift50:.1f}%")


🏆 Final Pipeline Results:


,Stage,nDCG@10,nDCG@20,nDCG@50,Recall@100,Recall@200,QPS
0,1. Retrieval (Hybrid 64d),0.447200,0.484611,0.548729,0.739259,0.812455,70.505264
1,2. Reranking (Cross-Encoder),0.508085,0.539461,0.585389,NaN,NaN,5.756211



📈 Accuracy @10 Lift from Reranking: +13.6%

📈 Accuracy @20 Lift from Reranking: +11.3%

📈 Accuracy @50 Lift from Reranking: +6.7%


#### By using JUST BASIC models: "BAAI/bge-base-en-v1.5" for RETRIEVAL and "mixedbread-ai/mxbai-rerank-base-v1" for RERANKING, we managed to get acceptable results: Recall@200 = 81.25% and nDCG@20 = 53.95%.

#### With access to enterprise compute (e.g., NVIDIA A100s or H100s), these metrics can be aggressively pushed higher through three direct upgrades:
#### - Parameter Scaling with Larger models.
#### - Larger Batch Sizes which will increase the discriminative power of MNRL.
#### - Expanded Context Windows by increasing the max_seq_length to 512 or 1024.